In [68]:
import pandas

import pygsheets
import numpy
import scipy
import re

# import data directly from google sheets

In [69]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')
spreadsheet = gc.open_by_key('1MK6yVWDUnbfXRjynMQcxIaeUc666XIKeKdVwtK1bd8I')

gas_pipes = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

pipes_df_orig = oil_pipes.copy() #pandas.concat([gas_pipes, oil_pipes], ignore_index=True)

#get country ratios sheet
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

In [70]:
gas_fuel_options = ['Gas']
ngl_fuel_options = ['NGL', 
                    'NGL, oil products', 
                    #'Oil products (only)',
                    'Oil, NGL', 
                    'Oil, NGL, naphtha',
                    #'Naphtha (only)',
                    #'Naphtha, oil products',
                    'LPG', 
                    #'Condensate', 
                    #'Oil, oil products'
                   ]
oil_fuel_options = ['Oil',
                    #'Oil products' 
                    'Oil, NGL', 
                    'Oil, NGL, naphtha'
                   ]

In [71]:
status_list = ['proposed', 
               'construction', 
               'shelved', 
               'cancelled', 
               'operating', 
               'idle', 
               'mothballed', 
               'retired']
country_list = sorted(set(region_df_orig['Country'].tolist()))
region_list = sorted(set(region_df_orig['Region'].tolist()))
if '--' in region_list:
    region_list.remove('--')
subregion_list = sorted(set(region_df_orig['SubRegion'].tolist()))
if '--' in subregion_list:
    subregion_list.remove('--')

#country_list = sorted(list(set(terms_df_orig['Country'])))
#region_list = sorted(list(set(terms_df_orig['Region'])))

## replace "--" with NaN, removing empty rows

the dataset is structured to have -- wherever there's a lookup value that doesn't exist; replacing it with NaN (numpy.nan) allows pandas to treat it as a null value, which makes calculations much easier

In [73]:
# replace -- entries with NaN
pipes_df_orig = pipes_df_orig.replace('--', numpy.nan)
pipes_df_orig = pipes_df_orig[pipes_df_orig['PipelineName']!='']

missing_wiki_projectids = pipes_df_orig.loc[pipes_df_orig.Wiki==''].ProjectID.tolist()
pipes_df_orig = pipes_df_orig[pipes_df_orig['Wiki']!='']

country_ratios_df.replace('--', numpy.nan, inplace=True)
#country_ratios_df = country_ratios_df.loc[~country_ratios_df.ProjectID.isin(missing_wiki_projectids)]

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/2390992745.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pipes_df_orig = pipes_df_orig.replace('--', numpy.nan)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/2390992745.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_ratios_df.replace('--', numpy.nan, inplace=True)


# km by country, km by region calculations

In [74]:
dict_subregion_region = pandas.Series(region_df_orig.Region.values, index=region_df_orig.SubRegion).to_dict()
dict_subregion_region

{'--': '--',
 'Northern Africa': 'Africa',
 'Sub-Saharan Africa': 'Africa',
 'Latin America and the Caribbean': 'Americas',
 'Northern America': 'Americas',
 'Central Asia': 'Asia',
 'Eastern Asia': 'Asia',
 'South-eastern Asia': 'Asia',
 'Southern Asia': 'Asia',
 'Western Asia': 'Asia',
 'Eastern Europe': 'Europe',
 'Northern Europe': 'Europe',
 'Southern Europe': 'Europe',
 'Western Europe': 'Europe',
 'Australia and New Zealand': 'Oceania',
 'Melanesia': 'Oceania',
 'Micronesia': 'Oceania',
 'Polynesia': 'Oceania'}

In [75]:
region_df_orig_cleaned = region_df_orig.loc[(region_df_orig.Region!='--')&
                                            (region_df_orig.SubRegion!='--')]
multiindex_region_subregion = region_df_orig_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion_country = region_df_orig_cleaned.groupby(['Region','SubRegion','Country'])['Country'].count().index

In [76]:
country_ratios_fuel_df = country_ratios_df[country_ratios_df.Fuel.isin(oil_fuel_options)]

km_by_country_df = pandas.DataFrame(columns=status_list, index=country_list)
km_by_subregion_df = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)
km_by_region_df = pandas.DataFrame(columns=status_list, index=region_list)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_fuel_df_status = country_ratios_fuel_df[country_ratios_fuel_df['Status']==status]
    km_by_country_df[status] = country_ratios_fuel_df_status.groupby('Country')['LengthMergedKmByCountry'].sum()
    km_by_subregion_df[status] = country_ratios_fuel_df_status.groupby(['Region','SubRegion'])['LengthMergedKmByCountry'].sum()
    km_by_region_df[status] = country_ratios_fuel_df_status.groupby('Region')['LengthMergedKmByCountry'].sum()

# # fill NaN with 0.0
km_by_subregion_df = km_by_subregion_df.fillna(0)
km_by_country_df = km_by_country_df.fillna(0)
km_by_region_df = km_by_region_df.fillna(0)

#km_by_region_df.sort_index(level='Region', inplace=True)
#km_by_region_df = km_by_region_df.loc[~(km_by_region_df==0).all(axis=1)]

# total
# km_by_region_df.loc['Total',:] = km_by_region_df.sum(axis=0).values
# km_by_country_df.loc['Total',:] = km_by_country_df.sum(axis=0).values

km_by_subregion_df['proposed+construction'] = km_by_subregion_df[['proposed','construction']].sum(axis=1)
km_by_subregion_df = km_by_subregion_df[['proposed', 'construction', 'proposed+construction', 'shelved', 'cancelled', 'operating', 'idle', 'mothballed', 'retired']]

km_by_country_df['proposed+construction'] = km_by_country_df[['proposed','construction']].sum(axis=1)
km_by_country_df.sort_values('proposed+construction', ascending=False, inplace=True)
km_by_country_df = km_by_country_df.loc[~(km_by_country_df==0).all(axis=1)]
km_by_country_df.loc[:,'Region'] = region_df_orig.set_index('Country').loc[km_by_country_df.index.tolist()].Region
km_by_country_df.loc[:,'Subregion'] = region_df_orig.set_index('Country').loc[km_by_country_df.index.tolist()].SubRegion
km_by_country_df = km_by_country_df[['Region','Subregion','proposed', 'construction', 'proposed+construction', 'shelved', 'cancelled', 'operating', 'idle', 'mothballed', 'retired']]
km_by_country_df = km_by_country_df.loc[(km_by_country_df.Region!='--')&
                                        (km_by_country_df.Subregion!='--')]

km_by_region_df['proposed+construction'] = km_by_region_df[['proposed','construction']].sum(axis=1)
km_by_region_df = km_by_region_df[['proposed', 'construction', 'proposed+construction', 'shelved', 'cancelled', 'operating', 'idle', 'mothballed', 'retired']]

km_by_subregion_df.index.set_names(['Region','Subregion'], inplace=True)
km_by_subregion_df.loc['Total',:] = km_by_subregion_df.sum(axis=0).values
km_by_subregion_df.replace(0,'', inplace=False)

===country-level calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired


proposed construction  \
Region   Subregion                                                 
Africa   Northern Africa                    768.500      100.000   
         Sub-Saharan Africa               7,081.260                
Americas Latin America and the Caribbean  2,417.000      653.000   
         Northern America                 1,094.010      270.710   
Asia     Central Asia                       264.790                
         Eastern Asia                     6,965.530    1,446.150   
         South-eastern Asia                 228.020                
         Southern Asia                       69.470    5,292.000   
         Western Asia                     5,875.560    2,657.000   
Europe   Eastern Europe                     921.980    1,894.140   
         Northern Europe                                           
         Southern Europe                    230.440                
         Western Europe                                            
Oceania  Australia and New Zealand                                 
         Melanesia                                                 
         Micronesia                                                
         Polynesia                                                 
Total                                    25,916.560   12,313.000   

                                         proposed+construction    shelved  \
Region   Subregion                                                          
Africa   Northern Africa                               868.500    488.000   
         Sub-Saharan Africa                          7,081.260    498.130   
Americas Latin America and the Caribbean             3,070.000              
         Northern America                            1,364.720  5,669.040   
Asia     Central Asia                                  264.790              
         Eastern Asia                                8,411.680    154.000   
         South-eastern Asia                            228.020              
         Southern Asia                               5,361.470              
         Western Asia                                8,532.560  1,544.610   
Europe   Eastern Europe                              2,816.120    585.260   
         Northern Europe                                                    
         Southern Europe                               230.440  1,034.630   
         Western Europe                                           149.260   
Oceania  Australia and New Zealand                                          
         Melanesia                                                          
         Micronesia                                                         
         Polynesia                                                          
Total                                               38,229.560 10,122.930   

                                          cancelled   operating      idle  \
Region   Subregion                                                          
Africa   Northern Africa                    620.040  17,600.620             
         Sub-Saharan Africa               1,500.000   9,407.940             
Americas Latin America and the Caribbean  3,937.830  26,843.530             
         Northern America                34,811.950 113,877.740             
Asia     Central Asia                     1,099.030   9,207.620             
         Eastern Asia                     7,230.680  34,607.170    46.350   
         South-eastern Asia                 306.000   1,649.520             
         Southern Asia                    6,780.040  26,218.240             
         Western Asia                     1,673.640  26,513.570   837.000   
Europe   Eastern Europe                   4,798.560  56,847.050 2,434.000   
         Northern Europe                              6,181.950             
         Southern Europe                  1,340.660   3,000.350   123.000   
         Western Europe                     634.250  11,048.820

In [77]:
#km_by_country_df.to_excel('km-by-country-region-subregion.xlsx')
km_by_country_df

,Region,Subregion,proposed,construction,proposed+construction,shelved,cancelled,operating,idle,mothballed,retired
China,Asia,Eastern Asia,"6,378.230","1,446.150","7,824.380",154.000,"7,230.680","34,559.130",46.350,0.000,"7,588.550"
Iraq,Asia,Western Asia,"2,855.040","1,154.590","4,009.630",0.000,898.800,"8,346.520",0.000,959.070,0.000
India,Asia,Southern Asia,0.000,"2,824.000","2,824.000",0.000,"1,338.000","9,254.170",0.000,0.000,0.000
Syria,Asia,Western Asia,"2,140.410",0.000,"2,140.410",0.000,0.000,169.510,0.000,880.250,217.110
Iran,Asia,Southern Asia,57.000,"2,028.000","2,085.000",0.000,"1,536.400","15,828.280",0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...
France,Europe,Western Europe,0.000,0.000,0.000,0.000,0.000,"4,696.810",257.610,0.000,0.000
Gabon,Africa,Sub-Saharan Africa,0.000,0.000,0.000,0.000,0.000,475.000,0.000,0.000,0.000
Georgia,Asia,Western Asia,0.000,0.000,0.000,0.000,0.000,247.540,386.720,0.000,0.000
Germany,Europe,Western Europe,0.000,0.000,0.000,149.260,458.920,"3,433.530",14.080,0.000,0.000


In [78]:
km_by_country_df.sort_values('construction', ascending=False).to_excel('km-by-country-region-subregion-sorted-by-construction.xlsx')
#km_by_country_df

In [79]:
km_by_region_df

,proposed,construction,proposed+construction,shelved,cancelled,operating,idle,mothballed,retired
Africa,"7,849.760",100.000,"7,949.760",986.130,"2,120.040","27,008.560",0.000,0.000,0.000
Americas,"3,511.010",923.710,"4,434.720","5,669.040","38,749.780","140,721.270",0.000,0.000,"6,453.810"
Asia,"13,403.370","9,395.150","22,798.520","1,698.610","17,089.390","98,196.120",883.350,"4,023.840","9,374.090"
Europe,"1,152.420","1,894.140","3,046.560","1,769.150","6,773.470","77,078.170","2,828.690","1,370.620","1,525.900"
Oceania,0.000,0.000,0.000,0.000,0.000,"2,115.300",0.000,0.000,187.000


# projects by country, by region

In [80]:
country_ratios_fuel_df = country_ratios_df[country_ratios_df.Fuel.isin(oil_fuel_options)]

num_by_country_df = pandas.DataFrame(columns=status_list, index=country_list)
num_by_region_df = pandas.DataFrame(columns=status_list, index=region_list)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_fuel_df_status = country_ratios_fuel_df[country_ratios_fuel_df['Status']==status]
    num_by_country_df[status] = country_ratios_fuel_df_status.groupby('ProjectID')['LengthMergedKmByCountry'].count()

print('===regional calculations===')
for status in status_list:
    print(status)
    country_ratios_fuel_df_status = country_ratios_fuel_df[country_ratios_fuel_df['Status']==status]
    num_by_region_df[status] = country_ratios_fuel_df_status.groupby('Region')['LengthMergedKmByCountry'].count()

# # fill NaN with 0.0
num_by_region_df = num_by_region_df.fillna(0)
num_by_country_df = num_by_country_df.fillna(0)

===country-level calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired
===regional calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired


In [81]:
num_by_region_df

,proposed,construction,shelved,cancelled,operating,idle,mothballed,retired
Africa,13.000,1.000,2.000,4.000,109,0.000,0.000,0.000
Americas,23.000,6.000,3.000,55.000,382,0.000,0.000,5.000
Asia,57.000,51.000,4.000,26.000,526,3.000,12.000,33.000
Europe,8.000,6.000,9.000,24.000,201,14.000,5.000,4.000
Oceania,0.000,0.000,0.000,0.000,12,0.000,0.000,1.000


# numbers of pipes finished in a given year

In [82]:
# count pipelines that are already operating
pipes_started = pipes_df_orig.copy()
pipes_started['StartYearEarliest'].replace(numpy.nan,'',inplace=True)
pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started.Fuel.isin(oil_fuel_options))]
pipes_started_sum = pipes_started.groupby('StartYearEarliest')['LengthMergedKm'].sum()

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/1020291388.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pipes_started['StartYearEarliest'].replace(numpy.nan,'',inplace=True)


In [83]:
# count km of pipeline for each start year
pipes_started_startyear = pipes_df_orig.copy()
pipes_started_startyear['StartYearEarliest'].replace(numpy.nan,'',inplace=True)
pipes_started_startyear = pipes_started_startyear[(pipes_started_startyear['Status'].isin(['operating'])) &
                              (pipes_started_startyear.Fuel.isin(oil_fuel_options))]
#pipes_started_startyear_sum = pipes_started_startyear.groupby('StartYearLatest')['LengthMergedKm'].sum()

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/1514101266.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pipes_started_startyear['StartYearEarliest'].replace(numpy.nan,'',inplace=True)


In [84]:
# indev pipelines = proposed or construction or shelved
pipes_indev = pipes_df_orig.copy()
pipes_indev = pipes_indev.loc[(pipes_indev.Fuel.isin(oil_fuel_options))&
                              (pipes_indev.Status.isin(['proposed','construction','shelved']))].groupby('Status')['LengthMergedKm'].sum()
#pipes_indev = pipes_indev.loc[(pipes_indev.StartYearEarliest>2022)|
#                              (pipes_indev.StartYearEarliest.isna())].groupby('Status')['LengthMergedKm'].sum()

In [85]:
pipes_indev_to2030 = pipes_df_orig.copy()
pipes_indev_to2030 = pipes_indev_to2030.loc[(pipes_indev_to2030.Fuel.isin(oil_fuel_options)) &
                                        (pipes_indev_to2030['Status'].isin(['proposed','construction','shelved'])) &
                                        (pipes_indev_to2030['StartYearEarliest'].isin(list(range(2023,2031))))]
pipes_indev_to2030 = pipes_indev_to2030.groupby('Status')['LengthMergedKm'].sum()


In [86]:
pipes_indev_startyear = pipes_df_orig.copy()
pipes_indev_startyear = pipes_indev_startyear[(pipes_indev_startyear['Status'].isin(['proposed','construction','shelved'])) &
                                              (pipes_indev_startyear.Fuel.isin(oil_fuel_options))]
pipes_indev_startyear = pipes_indev_startyear.groupby(['Status','StartYearEarliest'], dropna=False)['LengthMergedKm'].sum(min_count=1)

In [87]:
pipes_indev_startyear.unstack()

StartYearEarliest,"1,974.000","2,018.000","2,021.000","2,023.000","2,024.000","2,025.000","2,026.000","2,027.000","2,029.000","2,030.000","2,033.000",NaN
Status,,,,,,,,,,,,
construction,NaN,305.000,NaN,NaN,"3,478.380","1,263.470","1,540.000",NaN,NaN,NaN,NaN,"5,135.900"
proposed,523.040,NaN,NaN,NaN,"2,677.780","2,106.400","1,943.710",800.290,"1,249.000",NaN,NaN,"16,415.090"
shelved,NaN,NaN,804.670,536.000,NaN,160.000,NaN,600.000,NaN,"4,800.000",NaN,"3,222.020"


In [88]:
pipes_indev_startyear['construction',2024]

np.float64(3478.38)

## 2023–2030 figure, go back to 2008

## percent pipelines with start date (sanity check)

In [89]:
pipes_scratch = pipes_df_orig.copy()
pipes_scratch = pipes_scratch[(pipes_scratch.Fuel.isin(oil_fuel_options)) &
                              (pipes_scratch['Status'].isin(['proposed','construction','shelved']))]

In [90]:
pipes_scratch[~pipes_scratch['StartYearEarliest'].isnull()]['StartYearEarliest'].count()/pipes_scratch.shape[0]


np.float64(0.3732394366197183)

## regional pipelines started in given year

In [91]:
pipes_df_orig['StartYearEarliest'].min()

np.float64(1928.0)

In [92]:
years_array = numpy.arange(float(pipes_df_orig['StartYearEarliest'].min()), 
                           float(pipes_df_orig['StartYearEarliest'].max())+1)
regions_startyear_sums_df = pandas.DataFrame(numpy.nan, index=years_array, columns=region_list)

In [93]:
#pipes_started = pipes_df.copy()[(pipes_df['Status'].isin(['operating'])) & pipes_df['Fuel']=='oil']
#pipes_started_sum = pipes_started.groupby('StartYearLatest')['LengthMergedKm'].sum()

for region in region_list:
    pipes_started = pipes_df_orig.copy()[(pipes_df_orig['Status'].isin(['operating','retired','idle'])) & 
                                    (pipes_df_orig.Fuel.isin(oil_fuel_options))]
    pipes_started = pipes_started[pipes_started['StartRegion']==region]
    pipes_started_sum_up = pipes_started.groupby('StartYearEarliest')['LengthMergedKm'].sum()
    
    regions_startyear_sums_df[region] = pipes_started_sum_up

### count fraction of available capacity information

count

In [94]:
pipes_df_subset = pipes_df_orig.loc[(pipes_df_orig.Status.isin(['construction','proposed']))&
                                     (pipes_df_orig.Fuel.isin(oil_fuel_options))]
pipes_df_subset.loc[~pipes_df_subset['CapacityBOEd'].isna()].groupby('StartRegion')['CapacityBOEd'].size()
#pipes_df_subset.groupby('StartRegion')['CapacityBOEd'].size()

StartRegion
Africa       7
Americas    20
Asia        47
Europe      19
Name: CapacityBOEd, dtype: int64

fraction

In [95]:
median_capacity = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options)]['CapacityBOEd'].median()

In [96]:
abs_dist_from_med_capacity = abs(pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options)]['CapacityBOEd']-
                                 pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options)]['CapacityBOEd'].median()).median()

In [97]:
pipes_df_orig['CapacityBOEd'].mean()

np.float64(310747.7796577629)

In [98]:
pipes_df_orig['CapacityBOEd'].std()

np.float64(405716.0844316151)

In [99]:
print(median_capacity)
print(abs_dist_from_med_capacity)

200000.0
130000.0


median capacity for a pipeline is about 4.14 bcm/y

median abs dist from the median is about 3.5.

# cost estimates (pipeline cost per km)

## pick out high and low quantiles

In [100]:
temp_df = pipes_df_orig.loc[(~pipes_df_orig.CostUSDPerKm.isnull())&
                            (pipes_df_orig.Fuel.isin(oil_fuel_options))]
qlo_val = 0.025
qhi_val = 0.975

q_lo=temp_df['CostUSDPerKm'].quantile(qlo_val)
q_hi=temp_df['CostUSDPerKm'].quantile(qhi_val)
print(temp_df['CostUSDPerKm'].quantile(qlo_val))
print(temp_df['CostUSDPerKm'].quantile(qhi_val))

temp_df = temp_df.loc[temp_df['CostUSDPerKm'].between(q_lo, q_hi, inclusive='neither')]

38149.657245
21012370.76149997


In [101]:
# pull out only pipelines that have a KNOWN length AND a cost
country_ratios_with_length_and_cost_df = country_ratios_df.loc[(country_ratios_df.Fuel.isin(oil_fuel_options)) & 
                                                               (country_ratios_df['CostUSDPerKm'].notna()) & 
                                                               (country_ratios_df['LengthKnownKmByCountry'].notna()) #&
                                                               #(country_ratios_df['LengthKnownKm']!=0) &
                                                               #(country_ratios_df['CostUSDPerKm']<10e6)
                                                              ]

country_ratios_with_length_and_cost_df = country_ratios_with_length_and_cost_df.loc[
    country_ratios_with_length_and_cost_df['CostUSDPerKm'].between(q_lo, q_hi, inclusive='neither')]
#country_ratios_with_length_and_cost_df = country_ratios_with_length_and_cost_df[~country_ratios_with_length_and_cost_df.ProjectID.isin(outliers_projectids)]

In [102]:
country_ratios_df.loc[(country_ratios_df.Fuel.isin(oil_fuel_options)) & 
                    (country_ratios_df['CostUSDPerKm'].notna()) & 
                    (country_ratios_df['LengthKnownKmByCountry'].notna())].shape

(203, 36)

In [103]:
country_ratios_df.loc[(country_ratios_df.Fuel.isin(oil_fuel_options))].shape

(1601, 36)

### global mean value

In [104]:
global_mean = country_ratios_with_length_and_cost_df['CostUSDPerKm'].drop_duplicates().mean()
country_ratios_with_length_and_cost_df['CostUSDPerKm'].drop_duplicates().mean()

np.float64(2972315.1358783785)

### calculate regional costs

In [105]:
region_list

['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']

In [106]:
pipes_costs_region_df = pandas.DataFrame(0, index=region_list, columns=['CostUSDPerKm','DataPoints'])#,'NumberOfLengths'])

for region in region_list:
    country_ratios_region_df = country_ratios_with_length_and_cost_df.loc[country_ratios_with_length_and_cost_df['Region']==region,:]
    pipes_costs_region_df.loc[region,'CostUSDPerKm'] = country_ratios_region_df['CostUSDPerKm'].mean()
    pipes_costs_region_df.loc[region,'DataPoints'] = list(set(country_ratios_region_df['ProjectID'])).__len__()
    

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/466907183.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3672104.262352941' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pipes_costs_region_df.loc[region,'CostUSDPerKm'] = country_ratios_region_df['CostUSDPerKm'].mean()


In [107]:
pipes_costs_subregion_df = pandas.DataFrame(0, index=subregion_list, columns=['CostUSDPerKm','DataPoints'])#,'NumberOfLengths'])

for subregion in subregion_list:
    country_ratios_subregion_df = country_ratios_with_length_and_cost_df.loc[country_ratios_with_length_and_cost_df['SubRegion']==subregion,:]
    n_datapoints = list(set(country_ratios_subregion_df['ProjectID'])).__len__()
    if n_datapoints < 3:
        pipes_costs_subregion_df.loc[subregion,'DataPoints'] = list(set(country_ratios_subregion_df['ProjectID'])).__len__()
        pipes_costs_subregion_df.loc[subregion,'CostUSDPerKm'] = pipes_costs_region_df.loc[dict_subregion_region[subregion],'CostUSDPerKm']
    else:
        pipes_costs_subregion_df.loc[subregion,'DataPoints'] = list(set(country_ratios_subregion_df['ProjectID'])).__len__()
        pipes_costs_subregion_df.loc[subregion,'CostUSDPerKm'] = country_ratios_subregion_df['CostUSDPerKm'].mean()

pipes_costs_subregion_df.sort_values('CostUSDPerKm', ascending=False)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/2437797586.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1213941.655' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pipes_costs_subregion_df.loc[subregion,'CostUSDPerKm'] = country_ratios_subregion_df['CostUSDPerKm'].mean()


,CostUSDPerKm,DataPoints
Western Asia,"4,764,942.196",16
Sub-Saharan Africa,"4,137,772.485",5
Central Asia,"3,415,805.405",2
Eastern Asia,"3,338,176.852",53
Eastern Europe,"3,287,981.256",14
Northern America,"3,073,540.371",33
Northern Africa,"2,818,379.187",6
Latin America and the Caribbean,"2,754,893.192",14
Northern Europe,"2,367,532.547",0
Western Europe,"2,367,532.547",1


In [108]:
pipes_costs_region_df

,CostUSDPerKm,DataPoints
Africa,"3,672,104.262",11
Americas,"2,973,572.629",47
Asia,"3,415,805.405",83
Europe,"2,367,532.547",15
Oceania,"1,213,941.655",6


In [109]:
country_ratios_region_df['CostUSDPerKm'].mean()

np.float64(1213941.655)

In [110]:
pandas.options.display.float_format = '{:,.3f}'.format
temp_df = pipes_costs_region_df.copy()
temp_df['CostUSDPerKm'] = temp_df['CostUSDPerKm']/1e6
temp_df.sort_values('CostUSDPerKm', ascending=False)#.loc[region]['CostUSDPerKm']

,CostUSDPerKm,DataPoints
Africa,3.672,11
Asia,3.416,83
Americas,2.974,47
Europe,2.368,15
Oceania,1.214,6


In [111]:
pandas.options.display.float_format = '{:,.3f}'.format
temp_df = pipes_costs_subregion_df.copy()
temp_df['CostUSDPerKm'] = temp_df['CostUSDPerKm']/1e6
temp_df.sort_values('CostUSDPerKm', ascending=False)#.loc[region]['CostUSDPerKm']

,CostUSDPerKm,DataPoints
Western Asia,4.765,16
Sub-Saharan Africa,4.138,5
Central Asia,3.416,2
Eastern Asia,3.338,53
Eastern Europe,3.288,14
Northern America,3.074,33
Northern Africa,2.818,6
Latin America and the Caribbean,2.755,14
Northern Europe,2.368,0
Western Europe,2.368,1


# tables etc.

## table for stranded asset calculations

## country-level capex estimates

In [112]:
pipes_costs_region_df.sort_values('CostUSDPerKm', ascending=False)

,CostUSDPerKm,DataPoints
Africa,"3,672,104.262",11
Asia,"3,415,805.405",83
Americas,"2,973,572.629",47
Europe,"2,367,532.547",15
Oceania,"1,213,941.655",6


add a cost USD estimate column and estimate costs based on region and km by country

In [113]:
pipes_costs_subregion_df

,CostUSDPerKm,DataPoints
Australia and New Zealand,"1,213,941.655",6
Central Asia,"3,415,805.405",2
Eastern Asia,"3,338,176.852",53
Eastern Europe,"3,287,981.256",14
Latin America and the Caribbean,"2,754,893.192",14
Melanesia,"1,213,941.655",0
Micronesia,"1,213,941.655",0
Northern Africa,"2,818,379.187",6
Northern America,"3,073,540.371",33
Northern Europe,"2,367,532.547",0


In [114]:
country_ratios_df_specific_fuel = country_ratios_df.loc[country_ratios_df.Fuel.isin(oil_fuel_options)]
country_ratios_df_specific_fuel.reset_index(drop=True, inplace=True)
country_ratios_df_specific_fuel.loc[:,'CostUSDEstimate'] = numpy.nan

for idx,row in country_ratios_df_specific_fuel.iterrows():
    # calculate cost
    cntry = row.Country
    #region = row.Region
    subregion = row.SubRegion
    km_by_cntry = row.LengthMergedKmByCountry
    #country_ratios_df.loc[idx,'CostUSDEstimate'] = pipes_costs_region_df.loc[region, 'CostUSDPerKm'] * km_by_cntry
    country_ratios_df_specific_fuel.loc[idx,'CostUSDEstimate'] = pipes_costs_subregion_df.loc[subregion, 'CostUSDPerKm'] * km_by_cntry

# replace any known costs now
country_ratios_df_specific_fuel.loc[(~country_ratios_df_specific_fuel.LengthKnownKmByCountry.isna())&
                    (~country_ratios_df_specific_fuel.CostUSDPerKm.isna()),'CostUSDEstimate'] = \
country_ratios_df_specific_fuel.loc[(~country_ratios_df_specific_fuel.LengthKnownKmByCountry.isna())&
                    (~country_ratios_df_specific_fuel.CostUSDPerKm.isna()), 'LengthMergedKmByCountry'] * \
country_ratios_df_specific_fuel.loc[(~country_ratios_df_specific_fuel.LengthKnownKmByCountry.isna())&
                    (~country_ratios_df_specific_fuel.CostUSDPerKm.isna()), 'CostUSDPerKm']

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_25150/3748187952.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  country_ratios_df_specific_fuel.loc[:,'CostUSDEstimate'] = numpy.nan


In [115]:
country_ratios_df_specific_fuel.SubRegion.unique()

array(['Northern America', 'Latin America and the Caribbean',
       'Northern Africa', 'Sub-Saharan Africa', 'Western Asia',
       'Southern Asia', 'Eastern Europe', 'Central Asia', 'Eastern Asia',
       'Southern Europe', 'Western Europe', 'Northern Europe',
       'South-eastern Asia', 'Melanesia', 'Australia and New Zealand'],
      dtype=object)

In [116]:
capex_by_country_df = pandas.DataFrame(columns=status_list, index=country_list)
capex_by_region_df = pandas.DataFrame(columns=status_list, index=region_list)
capex_by_subregion_df = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_df_specific_fuel_status = country_ratios_df_specific_fuel.loc[country_ratios_df_specific_fuel.Status==status]
    country_ratios_df_specific_fuel_status = country_ratios_df_specific_fuel_status.loc[~country_ratios_df_specific_fuel_status.SubRegion.isnull()]
    capex_by_country_df[status] = country_ratios_df_specific_fuel_status.groupby('Country')['CostUSDEstimate'].sum()/1e9
    capex_by_region_df[status] = country_ratios_df_specific_fuel_status.groupby('Region')['CostUSDEstimate'].sum()/1e9
    capex_by_subregion_df[status] = country_ratios_df_specific_fuel_status.groupby(['Region','SubRegion'])['CostUSDEstimate'].sum()/1e9

# # fill NaN with 0.0
capex_by_region_df = capex_by_region_df.fillna(0)
capex_by_country_df = capex_by_country_df.fillna(0)
capex_by_subregion_df = capex_by_subregion_df.fillna(0)

capex_by_region_df['proposed+construction'] = capex_by_region_df[['proposed','construction']].sum(axis=1)
capex_by_region_df = capex_by_region_df[['proposed', 'construction', 'proposed+construction', 'shelved', 'cancelled', 'operating', 'idle', 'mothballed', 'retired']]
capex_by_region_df.loc['Total',:] = capex_by_region_df.sum(axis=0).values

capex_by_country_df['proposed+construction'] = capex_by_country_df[['proposed','construction']].sum(axis=1)
capex_by_country_df = capex_by_country_df[['proposed', 'construction', 'proposed+construction', 'shelved', 'cancelled', 'operating', 'idle', 'mothballed', 'retired']]
#capex_by_country_df.sort_values('construction', ascending=False, inplace=True)
capex_by_country_df.loc['Total',:] = capex_by_country_df.sum(axis=0).values

capex_by_subregion_df['proposed+construction'] = capex_by_subregion_df[['proposed','construction']].sum(axis=1)
capex_by_subregion_df = capex_by_subregion_df[['proposed', 'construction', 'proposed+construction', 'shelved', 'cancelled', 'operating', 'idle', 'mothballed', 'retired']]
capex_by_subregion_df.loc['Total',:] = capex_by_subregion_df.sum(axis=0).values

===country-level calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired


In [117]:
capex_by_region_df.replace(0,'')

,proposed,construction,proposed+construction,shelved,cancelled,operating,idle,mothballed,retired
Africa,29.002,0.282,29.283,3.437,7.954,79.856,,,
Americas,13.365,2.161,15.526,17.424,116.445,439.385,,,19.836
Asia,53.441,44.338,97.779,7.874,47.448,303.356,4.143,12.919,33.843
Europe,8.259,6.228,14.487,3.249,17.946,225.158,6.511,4.507,4.726
Oceania,,,,,,2.338,,,0.227
Total,104.067,53.009,157.076,31.983,189.793,"1,050.094",10.654,17.426,58.632


In [118]:
capex_by_subregion_df.index = capex_by_subregion_df.index.set_names(['Region','Subregion'])
capex_by_subregion_df.replace(0,'')

proposed construction  \
Region   Subregion                                               
Africa   Northern Africa                    2.193        0.282   
         Sub-Saharan Africa                26.809                
Americas Latin America and the Caribbean    9.756        1.390   
         Northern America                   3.609        0.771   
Asia     Central Asia                       0.904                
         Eastern Asia                      19.975        3.394   
         South-eastern Asia                 0.255                
         Southern Asia                      0.122        9.195   
         Western Asia                      32.185       31.750   
Europe   Eastern Europe                     8.052        6.228   
         Northern Europe                                         
         Southern Europe                    0.207                
         Western Europe                                          
Oceania  Australia and New Zealand                               
         Melanesia                                               
         Micronesia                                              
         Polynesia                                               
Total                                     104.067       53.009   

                                         proposed+construction shelved  \
Region   Subregion                                                       
Africa   Northern Africa                                 2.475   1.375   
         Sub-Saharan Africa                             26.809   2.061   
Americas Latin America and the Caribbean                11.146           
         Northern America                                4.380  17.424   
Asia     Central Asia                                    0.904           
         Eastern Asia                                   23.368   0.514   
         South-eastern Asia                              0.255           
         Southern Asia                                   9.317           
         Western Asia                                   63.935   7.360   
Europe   Eastern Europe                                 14.279   1.924   
         Northern Europe                                                 
         Southern Europe                                 0.207   0.971   
         Western Europe                                          0.353   
Oceania  Australia and New Zealand                                       
         Melanesia                                                       
         Micronesia                                                      
         Polynesia                                                       
Total                                                  157.076  31.983   

                                         cancelled operating   idle  \
Region   Subregion                                                    
Africa   Northern Africa                     1.748    48.593          
         Sub-Saharan Africa                  6.207    31.263          
Americas Latin America and the Caribbean     9.284    71.810          
         Northern America                  107.161   367.575          
Asia     Central Asia                        3.754    30.425          
         Eastern Asia                       24.063   108.414  0.155   
         South-eastern Asia                  0.380     1.748          
         Southern Asia                      11.276    42.865          
         Western Asia                        7.975   119.904  3.988   
Europe   Eastern Europe                     14.791   179.511  5.752   
         Northern Europe                              16.965          
         Southern Europe                     1.757     2.525  0.115   
         Western Europe                      1.398    26.158  0.643   
Oceania  Australia and New Zealand                     2.017          
         Melanesia                                     0.322          
         Micronesia             

In [119]:
capex_by_country_df = capex_by_country_df.loc[~(capex_by_country_df==0).all(axis=1)]
#capex_by_country_df.sort_values('proposed+construction', ascending=False, inplace=True)
capex_by_country_df.to_excel('capex-by-country.xlsx')
#capex_by_country_df.head()

# capex tree diagram

In [ ]:
subregion_string.Subregion

In [ ]:
subregion_values_df.iloc[0]

In [ ]:
bars

In [ ]:
subregion_values_df.iloc[0]['Subregion']

In [ ]:
width_pixels = 640 # 640 default
height_pixels = 450 # 450 default
fig = mp.figure(figsize=(width_pixels/72, height_pixels/72))
fig.canvas.draw()
axis_text_color = '0.5'

ax=fig.add_subplot(111)
ax.axis('off')

which_status = 'proposed+construction'
subregion_divider_lw = 4
region_divider_lw = 3
divider_lc = '1'

capex_by_subregion_df_nozero = capex_by_subregion_df.loc[capex_by_subregion_df[which_status]>1]
capex_by_region_df_nozero = capex_by_subregion_df_nozero.groupby('Region').sum()

# first normalize region sizes (computes the areas for the treemap)
normalized_region_values = squarify.normalize_sizes(capex_by_region_df_nozero.drop('Total')[which_status].values,
                                                    dx=100,
                                                    dy=100)

# squarify them now (computes where they are arranged)
squarified_region_values = squarify.squarify(normalized_region_values,
                                             x=0,
                                             y=0,
                                             dx=100,
                                             dy=100)

capex_by_subregion_df_treemap = capex_by_subregion_df_nozero.drop('Total')[[which_status]].reset_index(level=1)

ax_bbox = ax.get_position()
x_scale = ax_bbox.bounds[2]
y_scale = ax_bbox.bounds[3]

for i in range(len(squarified_region_values)):
    region_name = capex_by_region_df_nozero.index.tolist()[i]
    print(region_name)

    # create region dividers
    ax_new = fig.add_axes([squarified_region_values[i]['x']/100*y_scale+ax_bbox.bounds[0],
                           squarified_region_values[i]['y']/100*y_scale+ax_bbox.bounds[1],
                           squarified_region_values[i]['dx']/100*x_scale,
                           squarified_region_values[i]['dy']/100*y_scale],
                          frameon=True,
                          transform=ax.transAxes)

    # pull out subregion values
    subregion_values_df = capex_by_subregion_df_treemap.loc[region_name]
    
    if subregion_values_df[which_status].size == 1:
        subregion_values = [subregion_values_df[which_status]]
    else:
        subregion_values = subregion_values_df.sort_values(which_status, ascending=False)[which_status].values
    
    normalized_subregion_values = squarify.normalize_sizes(subregion_values,
                                                           dx=squarified_region_values[i]['dx'],
                                                           dy=squarified_region_values[i]['dy'])
    ax_new.axis('off')
    
    #subregion squarify plot
    squarify.plot(normalized_subregion_values,
                  norm_x=100,
                  norm_y=100,
                  ax=ax_new,
                  pad=False,
                  color=[treemap_colors[region_name]]*normalized_subregion_values.__len__(),
                  linewidth=subregion_divider_lw,
                  clip_on=False,
                  edgecolor=divider_lc)

    region_text = ax_new.text(
                s=region_name+'\n'+'US${x:,.0f} bn'.format(x=round(capex_by_region_df_nozero.loc[region_name,which_status],1)),
                x=list((2,2,2,2))[i],
                y=1,
                #transform=ax_new.transAxes,
                ha='left',
                va='bottom',
                #wrap=True,
                size=font_size_axis,
                color='1',
                weight=text_weight_bold)
    region_text.set_path_effects([matplotlib.patheffects.withStroke(linewidth=1, foreground='0.25')])

    # for each region plot with subregions, get the rectangles (bars) of the subregions
    # final entry of bars is the full box of the plot, I think, so use N-1 of them...
    bars = [rect for rect in ax_new.get_children() if isinstance(rect, matplotlib.patches.Rectangle)]
    for j in range(bars.__len__()-1):
        # get lower left, top right corner of boxes
        x0,y0,x1,y1 = bars[j].get_bbox().x0, bars[j].get_bbox().y0, bars[j].get_bbox().x1, bars[j].get_bbox().y1
        if bars.__len__()-1 == 1:
            subregion_string = subregion_values_df.Subregion
        else:
            subregion_string = subregion_values_df.iloc[j].Subregion
        text = ax_new.text(x=x1-2,y=y1-3,
                    s='\n'.join(
                        textwrap.wrap(
                        subregion_string,
                        10,
                        break_long_words=False)
                    ),
                    size=font_size_axis*0.7,
                    color='1',
                    ha='right',
                    va='top',
                    weight=600,
                    linespacing=1)

# fig.tight_layout()
sub_text = textwrap.fill('Estimated capital expenditures of crude oil transmission pipelines (proposed and under construction)',59)
tit_text = textwrap.fill('Asia and Africa will have the most expensive buildout, if all is completed',40)

gemplot_title_subtitle(ax=ax, fig=fig,
                       title_text=tit_text,
                       subtitle_text=sub_text,vertical_shift=0.03)

gemplot_footer(ax=ax, fig=fig,
               footer_text='Source: Global Oil Infrastructure Tracker',
               footer_position=0.75
              )

gemplot_axes(ax=ax, fig=fig)

fig.savefig('./figures/capex-treemap-horiz-text.png', 
            dpi=300,
            bbox_inches='tight',
            transparent=False)
fig.savefig('./figures/capex-km-treemap-horiz-text.pdf', 
            bbox_inches='tight',
            transparent=False)

In [ ]:
bars.__len__()-1

In [ ]:
subregion_values_df

In [ ]:
subregion_values_df.iloc[0].Subregion

In [ ]:
subregion_values_df.iloc[1].Subregion

# km by country, capex by country

In [ ]:
capex_by_country_df.loc['Sri Lanka']

In [ ]:
km_by_country_df.loc['Sri Lanka']

In [ ]:
capex_by_country_df.loc['Iraq']

In [ ]:
km_by_country_df.loc['Iraq']

In [ ]:
km_and_capex_by_country = km_by_country_df[['proposed+construction']].copy()
km_and_capex_by_country = km_and_capex_by_country.rename(columns={'proposed+construction':'proposed+construction km'})
km_and_capex_by_country['proposed+construction capex'] = capex_by_country_df[['proposed+construction']]
km_and_capex_by_country.replace(numpy.nan,'',inplace=True)
km_and_capex_by_country.to_excel('km-and-capex-by-country.xlsx')

In [ ]:
km_and_capex_by_country

## capex data - 15 leading countries by construction

In [ ]:
country_order

In [ ]:
km_by_country_df.sort_values('construction', ascending=False)[:15]

In [ ]:
nbars = 15
country_order = km_by_country_df.sort_values('proposed+construction', ascending=False).index.tolist()[:nbars]
capex_by_country_df.loc[country_order]

## print out country-level stranded assets for report discussion

## in-dev for each country in list

# us infrastructure km and cost for Gulf Coast export buildout specifically

# numbers for report

## amount already being constructed

In [ ]:
# fraction of pipelines under construction compared to all in development
# this number is different from below because it's skipping some pipelines
km_by_country_total = km_by_country_df.sum(axis=0)
km_by_country_total['construction']/km_by_country_total['proposed+construction']

## capacity/potential emissions in development globally

In [ ]:
pipes_df_subset.CapacityBOEd

## total number of pipelines in dev - disagrees a little bit with country_ratios version

In [ ]:
pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                  (pipes_df_orig.Status.isin(['proposed','construction']))]['LengthMergedKm'].sum()

## top 10 pipelines

## biggest pipelines that went into construction in 2022/2023

In [ ]:
biggest_pipeline_names_table = pandas.DataFrame(pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                                                                  (pipes_df_orig.Status.isin(['construction','operating']))&
                                                                  (pipes_df_orig.ConstructionYear.isin([2023,2024]))].groupby(['PipelineName','Status','Wiki'])['LengthMergedKm'].sum(min_count=1).sort_values(ascending=False)[:19])

biggest_pipeline_names_table

## biggest pipeline projects by name in China, rather than individual ProjectID

In [ ]:
biggest_pipeline_names_table = pandas.DataFrame(pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                                                                  (pipes_df_orig.Status.isin(['proposed','construction']))&
                                                                  (pipes_df_orig.Countries.str.contains('China'))].groupby(['PipelineName','Status','Wiki'])['LengthMergedKm'].sum(min_count=1).sort_values(ascending=False)[:19])

biggest_pipeline_names_table

In [ ]:
pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                (pipes_df_orig.Status.isin(['construction']))&
                (pipes_df_orig.Countries.str.contains('China'))].shape

In [ ]:
pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                (pipes_df_orig.Status.isin(['construction']))&
                (pipes_df_orig.Countries.str.contains('China'))].LengthMergedKm.mean()

## km of pipeline with 2023, 2024 start years

In [ ]:
# for the key points
print(pipes_indev_startyear.loc['construction',2024],
      pipes_indev_startyear.loc['construction',2025])

print(pipes_indev_startyear.loc['proposed',2024],
      pipes_indev_startyear.loc['proposed',2025])

## km of gas pipelines globally in development

note this doesn't match up exactly with regional length calculations

In [ ]:
pipes_df_calc = pipes_df_orig.copy()
pipes_df_calc.replace('--',numpy.nan,inplace=True)
pipes_df_calc[(pipes_df_calc['Status'].isin(['proposed','construction'])) &
        (pipes_df_calc.Fuel.isin(oil_fuel_options))]['LengthMergedKm'].sum()

In [ ]:
regional_km_sums_df = pandas.DataFrame(index=region_list, columns=status_list)

for status in status_list:
    regional_km_sums_df[status] = pipes_df_calc[(pipes_df_calc.Fuel.isin(oil_fuel_options)) &
                                                (pipes_df_calc['Status']==status)].groupby('StartRegion')['LengthMergedKm'].sum()

regional_km_sums_df['P+C'] = regional_km_sums_df[['proposed','construction']].sum(axis=1)
total_row = regional_km_sums_df.sum(axis=0)
total_row.name = 'Total'
#regional_km_sums_df.append(total_row)

## country shares analysis

In [ ]:
country_ratios_df[(country_ratios_df['Status'].isin(['proposed','construction'])) &
               (country_ratios_df.Fuel.isin(oil_fuel_options))]['MergedKmByCountry'].sum()

## num in dev globally, whether capacity expansions or not

In [ ]:
print("capacity expansion projects:",
    pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                  (pipes_df_orig.Status.isin(['proposed','construction']))&
                  (pipes_df_orig.RouteType.isin(['Capacity expansion only','Bidirectionality upgrade only']))].shape)

print("non-zero length projects:",
      pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                  (pipes_df_orig.Status.isin(['proposed','construction']))&
                  (~pipes_df_orig.RouteType.isin(['Capacity expansion only','Bidirectionality upgrade only']))].shape)

In [ ]:
print(pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                  (pipes_df_orig.Status.isin(['proposed','construction']))&
                  (pipes_df_orig.RouteType.isin(['Capacity expansion only','Bidirectionality upgrade only']))]['CapacityBOEd'])

print(pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                  (pipes_df_orig.Status.isin(['proposed','construction']))&
                  (pipes_df_orig.RouteType.isin(['Capacity expansion only','Bidirectionality upgrade only']))][['Countries','CapacityBOEd']])

# delay/difficulty to build analysis

### delays in 2023

In [ ]:
# how many have a start year of 2023 but haven't begun?
pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                    (pipes_df_orig.StartYearEarliest==2023)].groupby(['Status'])['LengthMergedKm'].sum()

In [ ]:
# how many have a start year of 2023 but haven't begun?
pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
                    (pipes_df_orig.StartYearEarliest==2024)].groupby(['Status','StartRegion'])['LengthMergedKm'].sum()

### timeline of pipeline completion

In [ ]:
# timeline of pipeline completion?
# need both proposal year and needs to be operational and have a start year
pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
    (pipes_df_orig.Status=='operating')&
    (~pipes_df_orig.StartYearEarliest.isnull())&
    (pipes_df_orig.ProposalYear!='')]

In [ ]:
pipes_df_gas_with_completion_data = pipes_df_orig.loc[(pipes_df_orig.Fuel.isin(oil_fuel_options))&
    (pipes_df_orig.Status=='operating')&
    (~pipes_df_orig.StartYearEarliest.isnull())&
    (pipes_df_orig.ProposalYear!='')&
    (pipes_df_orig.ProposalYear!=pipes_df_orig.StartYearEarliest)]

# km of projects built in past 5 years, 10 years?